In [ ]:
!pip install dagshub

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import sklearn
import numpy as np
import dagshub
dagshub.init(repo_owner='slomi23', repo_name='slomi23_ML_Assignment_1', mlflow=True)


Initialized MLflow to track repo "slomi23/slomi23_ML_Assignment_1"

Repository slomi23/slomi23_ML_Assignment_1 initialized!

In [ ]:
# read the train set
trainset=pd.read_csv("house-prices-advanced-regression-techniques/train.csv")
# read the test set
testset=pd.read_csv("house-prices-advanced-regression-techniques/test.csv")
#trainset.head(100)
trainset.shape

(1460, 81)

<h>Filling null values</h1>


In [ ]:
# sort it with non null value counts
print(trainset.count().sort_values(ascending=True).to_string())

PoolQC              7
MiscFeature        54
Alley              91
Fence             281
MasVnrType        588
FireplaceQu       770
LotFrontage      1201
GarageType       1379
GarageCond       1379
GarageFinish     1379
GarageYrBlt      1379
GarageQual       1379
BsmtExposure     1422
BsmtFinType2     1422
BsmtFinType1     1423
BsmtCond         1423
BsmtQual         1423
MasVnrArea       1452
Electrical       1459
Neighborhood     1460
BldgType         1460
Condition2       1460
Condition1       1460
Id               1460
LandContour      1460
Utilities        1460
LotConfig        1460
LandSlope        1460
HouseStyle       1460
OverallQual      1460
OverallCond      1460
MSSubClass       1460
ExterCond        1460
Foundation       1460
Exterior2nd      1460
ExterQual        1460
YearRemodAdd     1460
RoofStyle        1460
RoofMatl         1460
Exterior1st      1460
YearBuilt        1460
Heating          1460
CentralAir       1460
HeatingQC        1460
BsmtFinSF2       1460
BsmtUnfSF 

checking the features with most nulls, if null has a meaning (spoiler: it does here)

In [ ]:
trainset["PoolQC"].unique()
# nan means no pool, which shouldnt be dropped 
# since its important for price
trainset.fillna({"PoolQC":"No Pool"}, inplace=True)

trainset["Fence"].unique()
#nan means no fence, which shouldnt be dropped 
# since it's important for price
trainset.fillna({"Fence":"No Fence"}, inplace=True)

trainset["MiscFeature"].unique()
trainset.fillna({"MiscFeature":"None"}, inplace=True)

trainset["Alley"].unique()
trainset.fillna({"Alley":"No Alley"}, inplace=True)

trainset["FireplaceQu"].unique()
trainset.fillna({"FireplaceQu":"No Fireplace"}, inplace=True)

trainset["MasVnrType"].unique()
trainset.fillna({"MasVnrType":"None"}, inplace=True)



for col in trainset.columns:
    if trainset[col].mode()[0] == np.nan:
        trainset.drop(col, axis=1, inplace=True)


first, lets try simple devision of training set and validation set 80:20

In [ ]:
from sklearn.model_selection import train_test_split
trainsetLR=trainset.copy()
X = trainsetLR.drop("SalePrice", axis=1)
y = trainsetLR["SalePrice"]

X_train_LR, X_val_LR, y_train_LR, y_val_LR = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
X_train_LR["SalePrice"] = y_train_LR
X_val_LR["SalePrice"] = y_val_LR

print(X_train_LR["MSZoning"].unique())
print(X_val_LR["MSZoning"].unique())



['RL' 'RM' 'FV' 'RH' 'C (all)']
['RL' 'RM' 'C (all)' 'FV' 'RH']


first lets convert every categorical feature into numerical for linear regression, simply, lets assign each category to the mean of the target when grouped by that category

In [ ]:
def preprocess (X_train_LR, xx):
    cat_cols = X_train_LR.select_dtypes(include="object").columns

    for col in cat_cols:
        X_train_LR[col]=X_train_LR[col].fillna(X_train_LR[col].mode()[0])
    #now numerical columns

    """stuff_to_replace = ["RL", "RM", "FV", "RP", "RH", "A", "C", "I"]
    for col in X_train_LR.columns:
        if col not in cat_cols and X_train_LR[col] in stuff_to_replace:
            X_train_LR[col] = X_train_LR[col].replace(np.nan)"""

    for col in X_train_LR.columns:
        if col not in cat_cols:
            X_train_LR[col] = X_train_LR[col].fillna(X_train_LR[col].mean())

    cat_to_num_maps = {}

    for col in cat_cols:
        cat_to_num_maps[col] = xx.groupby(col)["SalePrice"].mean()


    for col in cat_cols:
        X_train_LR[col + "_num"] = X_train_LR[col].map(cat_to_num_maps[col])
        X_train_LR.drop(col, axis=1, inplace=True)

    if "Id" in X_train_LR.columns and "SalePrice" in X_train_LR.columns:
        X_train_LR.drop(columns=["Id", "SalePrice"], inplace=True)
    
    for col in X_train_LR.columns:
        X_train_LR[col] = X_train_LR[col].fillna(X_train_LR[col].mean())

    return X_train_LR

X_val_LR=preprocess(X_val_LR, X_train_LR)

X_train_LR=preprocess(X_train_LR, X_train_LR)



In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

def scale_and_RFE(X_train_LR, model_to_fit, y):
    scaler=StandardScaler()
    scaled = scaler.fit_transform(X_train_LR)

    X_train_LR_scaled = pd.DataFrame(
        scaled,
        columns=X_train_LR.columns,
        index=X_train_LR.index
    )

    modelLR = LinearRegression()
    rfe=RFE(modelLR, n_features_to_select=20, step=1)
    rfe.fit(model_to_fit, y)
    rfe_selected_features = model_to_fit.columns[rfe.support_].to_list()
    for i, feature in enumerate(rfe_selected_features, 1):
        print(f"{i}. {feature}")

    X_train_LR_scaled = X_train_LR_scaled[rfe_selected_features]
    return X_train_LR_scaled

X_train_LR_scaled= scale_and_RFE(X_train_LR, X_train_LR, y_train_LR)
X_val_LR_scaled= scale_and_RFE(X_val_LR, X_train_LR, y_train_LR)


1. MSSubClass
2. LotFrontage
3. OverallQual
4. OverallCond
5. YearRemodAdd
6. 1stFlrSF
7. 2ndFlrSF
8. BsmtFullBath
9. BsmtHalfBath
10. FullBath
11. HalfBath
12. BedroomAbvGr
13. KitchenAbvGr
14. TotRmsAbvGrd
15. Fireplaces
16. GarageYrBlt
17. GarageCars
18. ScreenPorch
19. MoSold
20. YrSold
1. MSSubClass
2. LotFrontage
3. OverallQual
4. OverallCond
5. YearRemodAdd
6. 1stFlrSF
7. 2ndFlrSF
8. BsmtFullBath
9. BsmtHalfBath
10. FullBath
11. HalfBath
12. BedroomAbvGr
13. KitchenAbvGr
14. TotRmsAbvGrd
15. Fireplaces
16. GarageYrBlt
17. GarageCars
18. ScreenPorch
19. MoSold
20. YrSold


X_train_LR is ready, now lets do mlflow log

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
mlflow.set_tracking_uri(
    "sqlite:////home/sandro/slomi23_ML_Assignment_1/mlflow.db"
)
mlflow.set_experiment("House Price Prediction")
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline.fit(X_train_LR, y_train_LR)
scalers = [
    StandardScaler(),
    MinMaxScaler(),
    None
]

param_grid = {
    'scaler': scalers,
    'model__fit_intercept': [True, False],
    'model__positive': [True, False]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    return_train_score=True
)


with mlflow.start_run(run_name="linear_regression_without_WOE_and_kfold"):

    model = LinearRegression()
    model.fit(X_train_LR_scaled, y_train_LR)

    # predictions
    train_preds = model.predict(X_train_LR_scaled)
    val_preds = model.predict(X_val_LR_scaled)

    # metrics
    train_rmse = np.sqrt(mean_squared_error(y_train_LR, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val_LR, val_preds))

    # log metrics
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_params(model.get_params())
    print(f"Train RMSE: {train_rmse}")
    print(f"Val RMSE: {val_rmse}")

    # log model
    mlflow.sklearn.log_model(model, "linear_regression_model")


2026/04/11 08:28:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 08:28:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train RMSE: 35086.943756313725
Val RMSE: 38615.30913031543


Train RMSE: 29260.53981371497 (around)
Val RMSE: 33823.117012592 (around)

now lets try kfold. still without WOE

In [ ]:
X_train_LR_KF = trainsetLR.copy()
X_train_LR_KF = preprocess(X_train_LR_KF, X_train_LR_KF)
X_train_LR_KF_scaled = scale_and_RFE(X_train_LR_KF, X_train_LR_KF, trainsetLR["SalePrice"])

y = trainsetLR["SalePrice"]

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
rmse_scores = []

with mlflow.start_run(run_name="linear_regression_with_kfold"):

    for train_idx, val_idx in kfold.split(X_train_LR_KF_scaled):

        X_train1, X_val = X_train_LR_KF_scaled.iloc[train_idx], X_train_LR_KF_scaled.iloc[val_idx]
        y_train1, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = LinearRegression()
        model.fit(X_train1, y_train1)

        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        rmse_scores.append(rmse)

        mlflow.log_metric(f"rmse_fold_{fold}", rmse)

        fold += 1

    mlflow.log_metric("rmse_mean", np.mean(rmse_scores))
    mlflow.log_metric("rmse_std", np.std(rmse_scores))

    final_model = LinearRegression()
    final_model.fit(X_train_LR_KF_scaled, y)

    mlflow.sklearn.log_model(final_model, "model")

    print(np.mean(rmse_scores))

2026/04/10 23:44:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


1. MSSubClass
2. OverallQual
3. OverallCond
4. YearBuilt
5. YearRemodAdd
6. 1stFlrSF
7. 2ndFlrSF
8. BsmtFullBath
9. BsmtHalfBath
10. FullBath
11. HalfBath
12. BedroomAbvGr
13. KitchenAbvGr
14. TotRmsAbvGrd
15. Fireplaces
16. GarageYrBlt
17. GarageCars
18. ScreenPorch
19. MoSold
20. YrSold


2026/04/10 23:44:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


36823.02490631259


Now lets try with One-hot Encoding and Kfold

In [ ]:
X_train_LR_OHE = trainsetLR.copy()
X_train_LR_OHE = X_train_LR_OHE.drop(columns=["Id", "SalePrice"])
y_train_LR_OHE = trainsetLR["SalePrice"]

cat_cols = X_train_LR_OHE.select_dtypes(include="object").columns
num_cols = X_train_LR_OHE.select_dtypes(exclude="object").columns

# categorical 
for col in cat_cols:
    X_train_LR_OHE[col] = X_train_LR_OHE[col].fillna(
        X_train_LR_OHE[col].mode()[0]
    )

# numeric 
for col in num_cols:
    X_train_LR_OHE[col] = X_train_LR_OHE[col].fillna(
        X_train_LR_OHE[col].mean()
    )
X_train_LR_OHE_dummies=pd.get_dummies(X_train_LR_OHE, columns=cat_cols,drop_first=True)
X_train_LR_OHE_dummies.head()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
rmse_scores = []
""""
with mlflow.start_run(run_name="linear_regression_with_kfold_and_OHE"):

    for train_idx, val_idx in kfold.split(X_train_LR_OHE_dummies):

        X_train1 = X_train_LR_OHE_dummies.iloc[train_idx]
        X_val = X_train_LR_OHE_dummies.iloc[val_idx]
        y_train1 = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = StandardScaler()

        X_train1_scaled = scaler.fit_transform(X_train1)
        X_val_scaled = scaler.transform(X_val)

        model = LinearRegression()
        model.fit(X_train1_scaled, y_train1)

        # predictions
        train_preds = model.predict(X_train1_scaled)
        val_preds = model.predict(X_val_scaled)

        # rmse
        train_rmse = np.sqrt(mean_squared_error(y_train1, train_preds))
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

        rmse_scores.append(val_rmse)

        # print per fold
        print(f"Fold {fold} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

        mlflow.log_metric(f"train_rmse_fold_{fold}", train_rmse)
        mlflow.log_metric(f"val_rmse_fold_{fold}", val_rmse)

        fold += 1

    mlflow.log_metric("rmse_mean", np.mean(rmse_scores))
    mlflow.log_metric("rmse_std", np.std(rmse_scores))

    final_model = LinearRegression()
    final_model.fit(X_train_LR_OHE_dummies, y)
    preds = final_model.predict(X_train_LR_OHE_dummies)
    final_rmse = np.sqrt(mean_squared_error(y, preds))

    print("Final training RMSE:", final_rmse)

    mlflow.sklearn.log_model(final_model, "model")

    print(np.mean(rmse_scores))"""




Fold 1 | Train RMSE: 0.0000 | Val RMSE: 0.0000
Fold 2 | Train RMSE: 0.0000 | Val RMSE: 0.0000
Fold 3 | Train RMSE: 0.0000 | Val RMSE: 0.0000
Fold 4 | Train RMSE: 0.0000 | Val RMSE: 0.0000


2026/04/11 09:14:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 09:14:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 5 | Train RMSE: 0.0000 | Val RMSE: 0.0000
Final training RMSE: 8.837276334712188e-11
2.4806111404258153e-10


now let's do decision tree

In [214]:
trainset_DT=trainset.copy()
X_DT = trainset_DT.copy()
y_DT = trainset_DT["SalePrice"]
X_DT = preprocess(X_DT, X_DT)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
rmse_scores = []
with mlflow.start_run(run_name="decision_tree_with_kfold"):
    for train_idx, val_idx in kfold.split(X_train_LR_KF_scaled):

        X_train1, X_val = X_DT.iloc[train_idx], X_DT.iloc[val_idx]
        y_train1, y_val = y_DT.iloc[train_idx], y_DT.iloc[val_idx]

        model = LinearRegression()
        model.fit(X_train1, y_train1)

        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        rmse_scores.append(rmse)

        mlflow.log_metric(f"rmse_fold_{fold}", rmse)

        fold += 1
        mlflow.log_metric("rmse_mean", np.mean(rmse_scores))
    mlflow.log_metric("rmse_std", np.std(rmse_scores))

    final_model = LinearRegression()
    final_model.fit(X_train_LR_KF_scaled, y)

    mlflow.sklearn.log_model(final_model, "model")

    print(np.mean(rmse_scores))

2026/04/11 00:47:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 00:47:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


34801.46939706734
